In [ ]:
import QuantLib as ql
import numpy as np
import pandas as pd
import random as rd
import os
import multiprocessing as mp
import time
import tqdm.notebook
import rbergomi
from py_vollib.black_scholes.implied_volatility import implied_volatility

In [ ]:
def simulate_rBergomi_paths(S0, Ks, T, rho, eta, H, xi_0, num_paths, num_steps):
    a = H - 0.5
    rB = rbergomi.rBergomi(n = num_steps, N = num_paths, T = T, a = a)
    dW1 = rB.dW1()
    dW2 = rB.dW2()
    Y = rB.Y(dW1)
    dB = rB.dB(dW1, dW2, rho = rho)
    V = rB.V(Y, xi = xi_0, eta = eta)
    S = rB.S(V, dB)
    ivs = dict()
    ST = S[:,-1][:,np.newaxis]
    for iK in Ks:
        call_payoff = np.maximum(ST - iK,0)
        call_price = np.mean(call_payoff, axis = 0)[:,np.newaxis].item()
        if call_price > 0 and call_price + iK > S0:
            iv = implied_volatility(call_price, S0, iK, T, 0.0, 'c')
            ivs[iK] = iv
    return ivs

In [ ]:
simulate_rBergomi_paths(1.0, [0.5, 1.2], 1.0, -0.5, 2.5, 0.2, 0.07, 10000, 100)

In [ ]:
def generateData(ID, 
                 runs, 
                 filename):

    S0 = 1.0
    r = 0.0
    num_paths = 30000
    num_steps = 100
    max_attempts = 10
    
    strikes = np.round(np.linspace(0.5,1.5,11),1)
    
    option_data = list()
    for run in range(runs):
        # Sample parameters
        startTime = time.process_time()
        attempts = 0
        ivs = dict()
        while attempts < max_attempts:
            attempts += 1
            try:
                eta = rd.uniform(0.5, 4.0)
                rho = rd.uniform(-0.95, -0.1)
                H = rd.uniform(0.025, 0.5)
                xi_0 = rd.uniform(0.01, 0.16)
                tau =  rd.uniform(0.1, 2.0)
                ivs = simulate_rBergomi_paths(S0, strikes, tau, rho, 
                                                eta, H, xi_0, num_paths, num_steps)
                break
            except:
                if attempts >= max_attempts:
                    print("Max attempts exceeded. Exiting.")
                    break
        endTime = time.process_time()
        elapsedTime = endTime - startTime
            
        for iv in ivs:    
            sample_data = list()
            sample_data.append(eta)
            sample_data.append(rho)
            sample_data.append(H)
            sample_data.append(xi_0)
            sample_data.append(tau)
            sample_data.append(iv)
            sample_data.append(ivs[iv])
            sample_data.append(elapsedTime)
            option_data.append(sample_data)

    heading_list = ['eta', 'rho', 'H', 'xi_0', 'tau', 'K', 'iv', 'time']
    df = pd.DataFrame(option_data, columns=heading_list)
    df.to_csv(filename + str(ID) + '.csv')
    print(ID)
    return(ID)

In [ ]:
generateData(1, 10, 'TestBergomiGrid')

In [ ]:
filename = 'bergomi_option_grid'
no_threads = os.cpu_count() - 4
pkgsize = 5000
dataSize = 20

In [ ]:
rd.seed()
pool = mp.Pool(processes=no_threads)
pbar = tqdm.tqdm(total=no_threads)
results = [pool.apply_async(generateData, args=(i,pkgsize,filename), 
                            callback=lambda _: pbar.update(1)) for i in range(dataSize)]
output = [p.get() for p in results]